## Ingesting data into bronze layer from raw ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
catalog_name = "practice_db_catalog"

In [0]:
df_listing = spark.read.format("json").load(f"/Volumes/{catalog_name}/airbnb/data_volume/airbnb/raw/listings.json")
display(df_listing.limit(10))
df_listing.write.format("delta").mode("overwrite").save(f"/Volumes/{catalog_name}/airbnb/data_volume/airbnb/bronze/listings_bronze")

In [0]:
from pyspark.sql.functions import explode
df_listing = spark.read.format("delta").load(f"/Volumes/{catalog_name}/airbnb/data_volume/airbnb/bronze/listings_bronze")
df_bronze_exploded = df_listing.withColumn("amenities", F.explode(F.col("amenities")))
display(df_listing_exploded.limit(50))

In [0]:
df_bronze = df_bronze_exploded.withColumn("has_parking",F.when(F.col("has_parking") == "Yes", True).when(F.col("has_parking") == "no", False).otherwise(F.col("has_parking")).cast("boolean"))
#df_bronze = df_bronze.withColumn("is_superhost",F.when(F.col("is_superhost") == "yes", True).when(F.col("is_superhost") == "no", False).otherwise(F.col("is_superhost")).cast("boolean"))
df_bronze = df_bronze.withColumn("is_superhost",F.col("is_superhost").cast("boolean"))
display(df_bronze.limit(50))


In [0]:
df_bronze = df_bronze.select("id","name","amenities","city","has_parking","is_superhost","created_date","last_booked_date","price_per_night")
df_bronze.write.format("delta").mode("overwrite").save(f"/Volumes/{catalog_name}/airbnb/data_volume/airbnb/silver/listings_silver")
